This notebook demonstrates the trajectory of a typical modeling process. We start out with a trivial model and increase complexity incrementally. With only a few lines of code, we achieve gains in predictive performance that exceed the existing relbench record.

In [ ]:
import logging
import pandas as pd

from relbench.datasets import get_dataset
from relbench.tasks import get_task
import getml

import lightgbm as lgb
from sklearn.metrics import roc_auc_score
import optuna


# optuna_hm-churn.py >> opt-hm-churn.log 2>&1 &

In [2]:
##############################################################################
#1. Set up dedicated logging
##############################################################################

logging.basicConfig(
    filename="opt-az-item-churn_n1_200.log",    # File where all logs will be written
    level=logging.INFO,             # You can switch to DEBUG for more detail
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S"
)
logger = logging.getLogger(__name__)

In [4]:
###############################################################################
# 2. Setup baseline model
# ##############################################################################

dataset = get_dataset("rel-amazon", download=True)
task = get_task("rel-amazon", "item-churn", download=True)

getml.set_project("rel-amazon")

population_roles = getml.data.Roles(
    join_key=["product_id"],
    target=["churn"],
    time_stamp=["timestamp"],
)

subsets = ("train", "test", "val")
populations = {}
for subset in subsets:
    populations[subset] = getml.data.DataFrame.from_parquet(
        f"{task.cache_dir}/{subset}.parquet",
        name=f"population_{subset}",
        roles=population_roles,
    )

product_roles = getml.data.Roles(
    join_key=["product_id"],
    numerical=["price"],
    categorical=["category", "brand"]
)

product_df = pd.read_parquet(f"{dataset.cache_dir}/db/product.parquet")
product_df.category = product_df.category.apply(lambda x:str(x))  # Lists with category headers are crudely converted to strings and counted as categories
product = getml.DataFrame.from_pandas(product_df,
name = 'product',
roles=product_roles)

review_roles_minimal = getml.data.Roles(
    time_stamp=["review_time"],
    join_key=["product_id"],
    numerical= ["rating"],
    categorical=['verified']
) #Actual Review text is not used in this model

review = getml.data.DataFrame.from_parquet(f"{dataset.cache_dir}/db/review.parquet", name = 'review', roles = review_roles_minimal)


  Loading pipelines... ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100% • 00:00


Connected to project 'rel-amazon'.

A  vanilla datamodel is created.

In [10]:
dm = getml.data.DataModel(population=populations["train"].to_placeholder())

dm.add(
   product.to_placeholder(), review.to_placeholder()
)
dm.population.join(
    dm.product, on="product_id", relationship=getml.data.relationship.many_to_one  #Taking out the product table actually increases the AUC!!
)
dm.population.join(
    dm.review, on="product_id", time_stamps=("timestamp", "review_time")#, memory = getml.data.time.days(365)
)

container = getml.data.Container(**populations)
container.add(product, review)

The prediction pipeline features the default FastProp algorithm Feature Learner and an XGBoostClassifier as predictor

In [11]:
pipe = getml.Pipeline(
    tags=["task: item-churn"],
    data_model=dm,
    feature_learners=[getml.feature_learning.FastProp()],
    predictors=[getml.predictors.XGBoostClassifier()],
    loss_function=getml.feature_learning.loss_functions.CrossEntropyLoss,
)

In [13]:
pipe.fit(container.train)

Checking data model...

  Staging... ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100% • 00:00

  Staging... ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100% • 00:00


OK.

  Staging... ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100% • 00:00
  Retrieving features from cache... ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100% • 00:00
  FastProp: Building features... ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100% • 00:15


Trained pipeline.

Time taken: 0:00:15.640370.



Pipeline(data_model='population_train',
         feature_learners=['FastProp'],
         feature_selectors=[],
         include_categorical=False,
         loss_function='CrossEntropyLoss',
         peripheral=['review'],
         predictors=['XGBoostClassifier'],
         preprocessors=[],
         share_selected_features=0.5,
         tags=['task: item-churn', 'container-N7t8SA', 'container-N7t8SA'])

In [14]:
print(pipe.score(container.test))
print(pipe.score(container.val))

  Staging... ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100% • 00:00
  Preprocessing... ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100% • 00:00
  FastProp: Building features... ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100% • 00:02
    date time             set used   target   accuracy       auc   cross entropy
0   2025-01-14 14:20:47   train      churn      0.7322    0.8104          0.5187
1   2025-01-14 14:20:57   test       churn      0.7489    0.8193          0.496 
  Staging... ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100% • 00:00
  Preprocessing... ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100% • 00:00
  FastProp: Building features... ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100% • 00:02
    date time             set used   target   accuracy       auc   cross entropy
0   2025-01-14 14:20:47   train      churn      0.7322    0.8104          0.5187
1   2025-01-14 14:20:57   test       churn      0.7489    0.8193          0.496 
2   2025-01-14 14:20:59   val        churn      0.7407    0.8168        

The predictive performance that was achieved is rather substandard. In order to exploit tha information in the reviews, we increase the complexity of the data model. In particular we try to capture temporal effects during feature generation. To this end, we extend the data model by peripheral tables of varying degrees of memory. We introduce the built-in seasonal preprocessor that captures regularly recurring events and we extend the set of possible aggregations that the FastProp algorithm will have at its disposal to create features with. 

We will omit the predictor argument in the pipeline, instead we limit us for to purely creating features.

In [5]:
dm = getml.data.DataModel(population=populations["train"].to_placeholder())

dm.add(
   product.to_placeholder(), review.to_placeholder()
)
dm.population.join(
    dm.product, on="product_id", relationship=getml.data.relationship.many_to_one
)
dm.population.join(
    dm.review, on="product_id", time_stamps=("timestamp", "review_time")
)
dm.population.join(
    dm.review, on="product_id", time_stamps=("timestamp", "review_time"), memory = getml.data.time.days(365)
)
dm.population.join(
    dm.review, on="product_id", time_stamps=("timestamp", "review_time"), memory = getml.data.time.days(183)
)

container = getml.data.Container(**populations)
container.add(product, review)

In [6]:
additional_aggregations = {
    getml.feature_learning.aggregations.EWMA_1D,
    getml.feature_learning.aggregations.EWMA_7D,
    getml.feature_learning.aggregations.EWMA_30D,
    getml.feature_learning.aggregations.Q_1,
    getml.feature_learning.aggregations.Q_5,
    getml.feature_learning.aggregations.Q_10,
    getml.feature_learning.aggregations.Q_25,
    getml.feature_learning.aggregations.TIME_SINCE_FIRST_MINIMUM,
    getml.feature_learning.aggregations.TIME_SINCE_LAST_MINIMUM,
    getml.feature_learning.aggregations.TIME_SINCE_LAST_MAXIMUM,
    getml.feature_learning.aggregations.TIME_SINCE_FIRST_MAXIMUM,
}

pred_pipe = getml.Pipeline(
    tags=["task: item-churn"],
    data_model=dm,
    preprocessors=[getml.preprocessors.Seasonal()],
    feature_learners=[getml.feature_learning.FastProp(
        num_threads=16,
        n_most_frequent=1,
        num_features=1600,
        aggregation=(
            getml.feature_learning.FastProp.agg_sets.default
        ),
    )],
    # predictors=[getml.predictors.XGBoostClassifier(n_jobs=16)],
    loss_function=getml.feature_learning.loss_functions.CrossEntropyLoss,
    include_categorical=True
)

pred_pipe.fit(container.train, check = False)

⠙ Staging...                                            0% • --:--

  FastProp: Trying 1650 features... ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100% • 13:22

  Staging... ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100% • 00:00
  Preprocessing... ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100% • 00:41
  FastProp: Trying 1650 features... ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100% • 13:23


Trained pipeline.

Time taken: 0:14:05.326972.



Pipeline(data_model='population_train',
         feature_learners=['FastProp'],
         feature_selectors=[],
         include_categorical=True,
         loss_function='CrossEntropyLoss',
         peripheral=['product', 'review'],
         predictors=[],
         preprocessors=['Seasonal'],
         share_selected_features=0.5,
         tags=['task: item-churn', 'container-3neSXi'])

Now as we have created the features, we use lightgbm as the predictor. First we will transform the original data to the features we have just learned, and save them to disk for later use.

In [ ]:
train_df = pred_pipe.transform(container.train, df_name="train_transform").to_pandas()
val_df   = pred_pipe.transform(container.val,   df_name="val_transform").to_pandas()
test_df  = pred_pipe.transform(container.test,  df_name="test_transform").to_pandas()

train_df.to_parquet(f"features_transform_train_n1.parquet")
val_df.to_parquet(f"features_transform_val_n1.parquet")
test_df.to_parquet(f"features_transform_test_n1.parquet")

In [ ]:
train_df = pd.read_parquet("features_transform_train_n1.parquet")
val_df   = pd.read_parquet("features_transform_val_n1.parquet")
test_df  = pd.read_parquet("features_transform_test_n1.parquet")


y_train =  train_df["churn"]
y_val   =   val_df["churn"]
y_test  =   test_df["churn"]

In [ ]:
train_df_lgbm = train_df.drop(columns=["product_id","timestamp","churn"])
val_df_lgbm   = val_df.drop(columns=["product_id","timestamp","churn"])
test_df_lgbm  = test_df.drop(columns=["product_id","timestamp","churn"])

In [ ]:
# 1. (Optional) Convert object columns for LightGBM
def convert_object_columns(df):
    for col in df.select_dtypes(include=["object"]).columns:
        try:
            df[col] = df[col].astype(int)
        except (ValueError, TypeError):
            df[col] = df[col].astype("category")
    return df

# ------------------------------------------------------------------------------
# Step 1: Prepare data and train an initial LGBM model
# ------------------------------------------------------------------------------
X_train_lgb = convert_object_columns(train_df_lgbm.copy())
X_val_lgb   = convert_object_columns(val_df_lgbm.copy())
X_test_lgb  = convert_object_columns(test_df_lgbm.copy())

# Identify any categorical columns
categorical_cols = X_train_lgb.select_dtypes(include=["category"]).columns.tolist()

# Create LightGBM dataset objects
train_data = lgb.Dataset(
    X_train_lgb,
    label=y_train,
    categorical_feature=categorical_cols,
    free_raw_data=False
)
val_data = lgb.Dataset(
    X_val_lgb,
    label=y_val,
    categorical_feature=categorical_cols,
    free_raw_data=False
)

# Define basic LightGBM parameters (adjust as needed)
params = {
    "objective": "binary",
    "metric": "auc",
    "num_threads": 32,
    "learning_rate": 0.1,
    "max_depth": 5,
    "num_leaves": 15,
    "feature_fraction": 1.0,  # like colsample_bytree=1.0
    "bagging_fraction": 1.0,  # like subsample=1.0
    "lambda_l1": 0.0,         # like reg_alpha=0
    "lambda_l2": 1.0,         # like reg_lambda=1
    "min_data_in_leaf": 1     # similar to min_child_weight=1
}

# Train the initial model
model_lgb = lgb.train(
    params,
    train_data,
    num_boost_round=200,           # analogous to n_estimators
    valid_sets=[val_data],
    callbacks=[lgb.early_stopping(stopping_rounds=10, verbose=False)]
)

# Evaluate on the test set
pred_test_lgb = model_lgb.predict(X_test_lgb, num_iteration=model_lgb.best_iteration)
test_auc_lgb = roc_auc_score(y_test, pred_test_lgb)
print(f"Initial LGBM Test AUC: {test_auc_lgb:.4f}")

By carefully adjusting our data model, we increase the predictive performance to to AUV = 0.8270

Now, as a standard step to maximize performance, we will optimize the hyperparamters of the predictor. 
Instead of all the generated features, we will pick the 200 most important ones. They capture virtually the entire predictive performance, and reduce computation time during optimization.

In [ ]:
###############################################################################
# 3. Objective function (Optuna)
# ##############################################################################
def objective(trial):
    params = {
        "objective": "binary",
        "metric": "auc",
        "verbosity": -1,               # Suppress LightGBM's default console spam
        "bagging_freq": 1,
        "feature_pre_filter": False,
        "max_depth": trial.suggest_int("max_depth", 3, 11),
        "learning_rate": trial.suggest_float("learning_rate", 1e-3, 0.1, log=True),
        "num_leaves": trial.suggest_int("num_leaves", 2, 1024),
        "subsample": trial.suggest_float("subsample", 0.05, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.05, 1.0),
        "lambda_l1": trial.suggest_float("lambda_l1", 1e-9, 10.0, log=True),
        "lambda_l2": trial.suggest_float("lambda_l2", 1e-9, 10.0, log=True),
        "min_data_in_leaf": trial.suggest_int("min_data_in_leaf", 1, 100),
    }

    model = lgb.train(
        params,
        train_data,
        num_boost_round=2000,
        valid_sets=[val_data],
        callbacks=[
            lgb.early_stopping(stopping_rounds=50, verbose=False),
            lgb.log_evaluation(period=0),  # period=0 suppresses iteration logs
        ]
    )

    # Predict on validation set
    pred_val = model.predict(val_df, num_iteration=model.best_iteration)
    auc_val = roc_auc_score(y_val, pred_val)

    # Log the result for this trial
    logger.info(
        f"Trial {trial.number} finished with AUROC={auc_val:.6f}, params={trial.params}"
    )

    return auc_val

In [ ]:
###############################################################################
# 4. Create or load Optuna study using SQLite
# ##############################################################################
storage_url = "sqlite:///opt-az-item-churn_n1_200.db"
study_name  = "opt-az-item-churn_n1_200"

study = optuna.create_study(
    study_name=study_name,
    storage=storage_url,
    load_if_exists=True,
    direction="maximize",
)
logger.info("Starting Optuna study...")

In [ ]:
###############################################################################
# 5. Run optimization
# ##############################################################################
study.optimize(objective, n_trials=50)

best_params = study.best_params
best_params["objective"] = "binary"
best_params["metric"] = "auc"
best_params["feature_pre_filter"] = False

logger.info(f"Best trial found: {study.best_trial.number}")
logger.info(f"Best params: {best_params}")

In [ ]:
###############################################################################
# 6. Final model training with best parameters
# ##############################################################################
final_model = lgb.train(
    best_params,
    train_data,
    num_boost_round=2000,
    valid_sets=[val_data],
    callbacks=[
        lgb.early_stopping(stopping_rounds=50, verbose=False),
        lgb.log_evaluation(period=0),  # No iteration logs, only final result
    ]
)

pred_test = final_model.predict(test_df, num_iteration=final_model.best_iteration)
test_auc = roc_auc_score(y_test, pred_test)
logger.info(f"Final model Test AUC: {test_auc:.6f}")

After Hyperopt we attain an Test AUC of XXX, which is marginally higher than the one of relbench. That means solely by diligently adjusting the data model, picking prudent preprocessors and extending the set of permissible aggregations, you can achieve and exceed existing cutting edge Machine Learning algorithm. All you need to know is how to handle 1 framework: getML.

In [ ]:
###############################################################################
# 7. Script end
# ##############################################################################
logger.info("Study completed.")